In [1]:
# Random COCO Split

import numpy as np
import json

np.random.seed(42)

# Load COCO annotations
with open("soft_COCO_2017_val.json", "r") as f:
    coco_data = json.load(f)

# Get unique image IDs
image_ids = [img["id"] for img in coco_data["images"]]

print(f"Total images: {len(image_ids)}")

Total images: 5000


In [2]:
# split
train_ids = np.random.choice(image_ids, size=4000, replace=False) # 80 % for training
val_ids = np.setdiff1d(image_ids, train_ids)  # remaining 20 % for validation

coco_split = {"train_imgs": train_ids.tolist(), "val_imgs": val_ids.tolist()}

with open("splits/COCO_train_val_split.json", "w") as f:
    json.dump(coco_split, f)

print(f"Train images: {len(train_ids)}")
print(f"Validation images: {len(val_ids)}")

Train images: 4000
Validation images: 1000


In [ ]:
# remove zero-area bounding boxes from annotations
for obj in coco_data["objects"]:
    for ann in coco_data["objects"][obj]:
        xmin, ymin, xmax, ymax = ann["bbox"]

        if xmax <= xmin or ymax <= ymin:
            print(f"Removing zero-area bbox: {ann['bbox']} in image {obj}")
            coco_data["objects"][obj].remove(ann)

Removing zero-area bbox: [247, 365, 247, 368] in image 000000207728.jpg
No. annots before: 3
No. annots after: 2
Removing zero-area bbox: [434, 420, 434, 420] in image 000000125572.jpg
No. annots before: 42
No. annots after: 41
Removing zero-area bbox: [485, 336, 485, 342] in image 000000239773.jpg
No. annots before: 6
No. annots after: 5
Removing zero-area bbox: [48, 268, 48, 268] in image 000000147740.jpg
No. annots before: 18
No. annots after: 17
Removing zero-area bbox: [88, 310, 88, 310] in image 000000152740.jpg
No. annots before: 21
No. annots after: 20
Removing zero-area bbox: [111, 425, 115, 425] in image 000000288584.jpg
No. annots before: 21
No. annots after: 20
Removing zero-area bbox: [279, 6, 279, 6] in image 000000076468.jpg
No. annots before: 267
No. annots after: 266
Removing zero-area bbox: [10, 11, 10, 20] in image 000000133244.jpg
No. annots before: 46
No. annots after: 45
Removing zero-area bbox: [0, 0, 0, 422] in image 000000564280.jpg
No. annots before: 5
No. ann

In [8]:
with open("soft_COCO_2017_val.json", "w") as f:
    json.dump(coco_data, f)

In [9]:
# Create new annotation files

coco_train = coco_data.copy()
coco_train["images"] = [img for img in coco_data["images"] if img["id"] in train_ids] # filter down images
coco_train["objects"] = {obj: coco_data["objects"][obj] for obj in coco_data["objects"] if int(obj.split(".")[0]) in train_ids} # filter down objects by filename keys

coco_val = coco_data.copy()
coco_val["images"] = [img for img in coco_data["images"] if img["id"] in val_ids] # filter down images
coco_val["objects"] = {obj: coco_data["objects"][obj] for obj in coco_data["objects"] if int(obj.split(".")[0]) in val_ids} # filter down objects by filename keys

with open("soft_COCO_2017_val_train.json", "w") as f:
    json.dump(coco_train, f)
with open("soft_COCO_2017_val_val.json", "w") as f:
    json.dump(coco_val, f)


In [1]:
# PascalVOC
import json

with open("soft_PascalVOC_2012_detection_trainval.json", "r") as f:
    data = json.load(f)

with open("splits/PascalVOC_train_val_split.json", "r") as f:
    split = json.load(f)

In [2]:
train = data.copy()

del train["objects"] # remove for better order of entries in the json file

train["images"] = [] # Initialize an empty list for the training images
for dic in data["images"]:
    if dic["file_name"].split(".")[0] in split["train_imgs"]:
        train["images"].append(dic)

train["objects"] = {} # Initialize an empty dictionary for the training objects
for fname in data["objects"]:
    if fname.split(".")[0] in split["train_imgs"]:
        train["objects"][fname] = data["objects"][fname]

with open("soft_PascalVOC_2012_detection_train.json", "w") as f:
    json.dump(train, f)

In [3]:
val = data.copy()

del val["objects"] # remove for better order of entries in the json file

val["images"] = [] # Initialize an empty list for the validation images
for dic in data["images"]:
    if dic["file_name"].split(".")[0] in split["val_imgs"]:
        val["images"].append(dic)

val["objects"] = {} # Initialize an empty dictionary for the validation objects
for fname in data["objects"]:
    if fname.split(".")[0] in split["val_imgs"]:
        val["objects"][fname] = data["objects"][fname]

with open("soft_PascalVOC_2012_detection_val.json", "w") as f:
    json.dump(val, f)

In [1]:
# KITTI

import json

with open("soft_Kitti_2D_train.json", "r") as f:
    data = json.load(f)

with open("splits/Kitti_train_val_split.json", "r") as f:
    split = json.load(f)

In [3]:
# fix the following artefact: sometimes xmax is smaller than xmin, and sometimes ymax is smaller than ymin, which causes problems for training. We will swap the coordinates if this is the case.
for fname in data["objects"]:
    for obj in data["objects"][fname]:
        if obj["bbox"][0] > obj["bbox"][2]: # if xmin > xmax
            obj["bbox"][0], obj["bbox"][2] = obj["bbox"][2], obj["bbox"][0] # swap

            print(f"Swapped xmin and xmax for {fname} because {obj['bbox'][0]} > {obj['bbox'][2]}")
        if obj["bbox"][1] > obj["bbox"][3]: # if ymin > ymax
            obj["bbox"][1], obj["bbox"][3] = obj["bbox"][3], obj["bbox"][1] # swap

            print(f"Swapped ymin and ymax for {fname} because {obj['bbox'][1]} > {obj['bbox'][3]}")


In [4]:
with open("soft_Kitti_2D_train.json", "w") as f: # store image information
    json.dump(data, f)

In [2]:
from PIL import Image
images = []
for obj in data["objects"]:
    try:
        img = Image.open("/path/to/datasets/KITTI/train/images/" + obj + ".png")
    except FileNotFoundError: # if not in training set, it will be in validation set
        img = Image.open("/path/to/datasets/KITTI/val/images/" + obj.split(".")[0] + ".png") 
        
    images.append({
        "file_name": obj + ".png",
        "id": int(obj.split(".")[0]),
        "width": img.width,
        "height": img.height
    })

# sort by id
images.sort(key=lambda x: x["id"])

data["images"] = images # image information as usual in COCO style datasets

data["categories"] = [ 
    {"id": i, "name": cat} for i, cat in enumerate(data["classes"])
]

with open("soft_Kitti_2D_train.json", "w") as f: # store image information
    json.dump(data, f)

In [5]:
train = data.copy()

del train["objects"] # remove for better order of entries in the json file

train["images"] = [] # Initialize an empty list for the training images
for dic in data["images"]:
    if dic["file_name"].split(".")[0] in split["train_imgs"]:
        train["images"].append(dic)

train["objects"] = {} # Initialize an empty dictionary for the training objects
for fname in data["objects"]:
    if fname.split(".")[0] in split["train_imgs"]:
        train["objects"][fname] = data["objects"][fname]

with open("soft_Kitti_2D_train_train.json", "w") as f:
    json.dump(train, f)

In [6]:
val = data.copy()

del val["objects"] # remove for better order of entries in the json file

val["images"] = [] # Initialize an empty list for the val images
for dic in data["images"]:
    if dic["file_name"].split(".")[0] in split["val_imgs"]:
        val["images"].append(dic)

val["objects"] = {} # Initialize an empty dictionary for the validation objects
for fname in data["objects"]:
    if fname.split(".")[0] in split["val_imgs"]:
        val["objects"][fname] = data["objects"][fname]

with open("soft_Kitti_2D_train_val.json", "w") as f:
    json.dump(val, f)